In [6]:
import numpy as np
import time
import contextlib
import io

from raiselab.core import dvqe


# ============================================================
# BASIC HELPERS
# ============================================================

def _objective(A, b, c, x):
    """
    Compute f(x) = x^T A x + b^T x + c.
    """
    return float(x.T @ A @ x + b.T @ x + c)


def _symmetrize(A):
    """
    Symmetrize A because x^T A x only depends on the symmetric part.
    """
    return 0.5 * (A + A.T)


def _scale_qubo(Q, q):
    """
    Scale QUBO coefficients for DVQE numerical stability.
    Scaling does not change the minimizer.
    """
    scale = max(
        float(np.max(np.abs(Q))) if Q.size > 0 else 0.0,
        float(np.max(np.abs(q))) if q.size > 0 else 0.0,
        1.0
    )

    return Q / scale, q / scale, scale


# ============================================================
# FIRST MAPPING: x-PROBLEM TO alpha-PROBLEM
# ============================================================

def _build_alpha_qup(A, b, c, x, lb, ub, rho):
    """
    Build the normalized alpha-space quadratic problem.

    Original problem:
        min_x x^T A x + b^T x + c

    At outer iterate x^k, define:
        x_candidate = x^k + R alpha

    where:
        R = diag(ub - lb)

    Then:
        f(x^k + R alpha)
        = alpha^T A_alpha alpha + b_alpha^T alpha + c_alpha

    Bounds on alpha:
        alpha_L_i = max(-rho, (lb_i - x_i) / range_i)
        alpha_U_i = min( rho, (ub_i - x_i) / range_i)

    This guarantees x_candidate remains inside [lb, ub].
    """

    var_range = ub - lb
    R = np.diag(var_range)

    A_alpha = R.T @ A @ R
    b_alpha = 2.0 * R.T @ A @ x + R.T @ b
    c_alpha = _objective(A, b, c, x)

    alpha_lb = np.maximum(-rho, (lb - x) / np.maximum(var_range, 1e-12))
    alpha_ub = np.minimum(rho, (ub - x) / np.maximum(var_range, 1e-12))

    # Numerical protection
    alpha_lb = np.minimum(alpha_lb, alpha_ub)

    return A_alpha, b_alpha, c_alpha, alpha_lb, alpha_ub, var_range


# ============================================================
# SECOND MAPPING: alpha-PROBLEM TO LOCAL ONE-BIT QUBO
# ============================================================

def _build_local_alpha_qubo(
    A_alpha,
    b_alpha,
    c_alpha,
    alpha,
    alpha_delta,
    alpha_lb,
    alpha_ub
):
    """
    Build one local n-bit QUBO in alpha-space.

    At inner iterate alpha^j, each binary z_i chooses between:
        moving alpha_i down
        moving alpha_i up

    Local mapping:
        alpha_candidate_i = alpha_i - d_minus_i + (d_plus_i + d_minus_i) z_i

    where:
        d_plus_i  = min(alpha_delta_i, alpha_ub_i - alpha_i)
        d_minus_i = min(alpha_delta_i, alpha_i - alpha_lb_i)

    In vector form:
        alpha_candidate = y + P z

    Then:
        f_alpha(y + Pz) = z^T Q z + q^T z + constant
    """

    d_plus = np.minimum(alpha_delta, alpha_ub - alpha)
    d_minus = np.minimum(alpha_delta, alpha - alpha_lb)

    d_plus = np.maximum(d_plus, 0.0)
    d_minus = np.maximum(d_minus, 0.0)

    y = alpha - d_minus
    p = d_plus + d_minus

    P = np.diag(p)

    Q_local = P.T @ A_alpha @ P
    q_local = 2.0 * P.T @ A_alpha @ y + P.T @ b_alpha

    constant = float(y.T @ A_alpha @ y + b_alpha.T @ y + c_alpha)

    return Q_local, q_local, constant, y, p, d_plus, d_minus


def _decode_alpha_solution(z, y, p):
    """
    Decode DVQE binary solution into alpha candidate.
    """
    z = np.asarray(z, dtype=float)
    return y + p * z



# ============================================================
# CLASSICAL NON-BRUTE-FORCE QUBO SOLVER
# ============================================================

def _qubo_energy(Q, q, z):
    """
    Compute QUBO energy:
        E(z) = z^T Q z + q^T z
    where z is binary.
    """
    z = np.asarray(z, dtype=float)
    return float(z.T @ Q @ z + q.T @ z)


def _solve_qubo_classical_local_search(
    Q,
    q,
    num_restarts=20,
    max_sweeps=100,
    seed=None,
    initial_z=None
):
    """
    Classical non-brute-force QUBO solver using random-restart
    one-bit-flip local search.

    It does NOT enumerate all 2^n binary solutions.

    Parameters
    ----------
    Q : ndarray, shape (n,n)
        QUBO quadratic matrix.

    q : ndarray, shape (n,)
        QUBO linear vector.

    num_restarts : int
        Number of random starting points.

    max_sweeps : int
        Maximum number of full coordinate-flip sweeps per restart.

    seed : int or None
        Random seed.

    initial_z : ndarray or None
        Optional warm-start binary vector.

    Returns
    -------
    best_z : ndarray, shape (n,)
        Best binary solution found.

    best_energy : float
        Best QUBO energy found.

    info : dict
        Solver information.
    """

    Q = np.asarray(Q, dtype=float)
    q = np.asarray(q, dtype=float)

    n = q.shape[0]

    if Q.shape != (n, n):
        raise ValueError("Q must have shape (n,n).")

    rng = np.random.default_rng(seed)

    best_z = None
    best_energy = np.inf

    starts = []

    if initial_z is not None:
        z0 = np.asarray(initial_z, dtype=int)
        if z0.shape != (n,):
            raise ValueError("initial_z must have shape (n,).")
        starts.append(np.clip(z0, 0, 1))

    for _ in range(num_restarts):
        starts.append(rng.integers(0, 2, size=n, dtype=int))

    total_flips = 0

    for z_start in starts:

        z = z_start.copy()
        current_energy = _qubo_energy(Q, q, z)

        for _ in range(max_sweeps):

            improved = False

            # Random coordinate order helps avoid deterministic bad cycles
            for i in rng.permutation(n):

                z_trial = z.copy()
                z_trial[i] = 1 - z_trial[i]

                trial_energy = _qubo_energy(Q, q, z_trial)

                if trial_energy < current_energy:
                    z = z_trial
                    current_energy = trial_energy
                    improved = True
                    total_flips += 1

            if not improved:
                break

        if current_energy < best_energy:
            best_energy = current_energy
            best_z = z.copy()

    info = {
        "method": "classical_local_search",
        "num_restarts": num_restarts,
        "max_sweeps": max_sweeps,
        "best_energy": best_energy,
        "total_flips": total_flips,
    }

    return best_z.astype(int), best_energy, info
    
# ============================================================
# MAIN SOLVER: SCALE-ADAPTIVE DQUP
# ============================================================

def dqup(
    A,
    b=None,
    c=0.0,
    x0=None,
    lb=None,
    ub=None,

    # Outer x-space settings
    rho0=0.1,
    rho_decay=0.4,
    rho_tol=1e-9,
    obj_tol=1e-10,
    grad_tol=None,
    max_outer_iters=5,

    # NEW outer early-stop setting
    max_consecutive_outer_rejections=4,

    # Inner alpha-space settings
    alpha0=None,
    alpha_step0=None,
    alpha_step_fraction=1.0,
    alpha_step_decay=0.1,
    alpha_step_tol=1e-8,
    max_inner_iters=20,

    # NEW inner early-stop setting
    max_consecutive_inner_rejections=20,

    alpha_warm_start=True,
    alpha_warm_start_scale=0.5,

    # DVQE settings
    mode="distributed",
    init_type=2,
    depth=1,
    lr=0.08,
    max_iters=30,
    qpu_qubit_config=None,
    rel_tol=1e-4,
    num_shots=128,
    final_shots=1000,
    warm_start_population=4,
    warm_start_iters=5,
    warm_start_shots=64,
    seed=None,
    backend=None,
    energy_mode="cvar",
    cvar_alpha=0.2,

    # QUBO solver selection
    qubo_solver="dvqe",          # "dvqe" or "classical"

    # Classical QUBO solver settings
    classical_num_restarts=20,
    classical_max_sweeps=100,

    # Printing controls
    verbose=False,
    silent=True,

    # Output settings
    return_info=True
):
    """
    DQUP: Scale-adaptive sequential one-bit DVQE solver for bounded
    continuous quadratic unconstrained programming.

    Solves approximately:

        min_x  x^T A x + b^T x + c
        s.t.   lb <= x <= ub

    Main idea:
    ----------
    At outer iteration k, transform the x-problem into a normalized
    alpha-problem:

        x_candidate = x^k + diag(ub-lb) alpha

    where alpha is bounded in a trust region:

        alpha_i in [alpha_lb_i, alpha_ub_i]
        alpha_lb_i >= -rho
        alpha_ub_i <=  rho

    Then solve the alpha-problem using an inner sequential one-bit DVQE loop.

    Each inner DVQE call solves an n-bit QUBO, where n is the number of
    continuous variables. Therefore, the method avoids n*K full discretization.

    New early-stop behavior:
    ------------------------
    - max_consecutive_inner_rejections:
        Stop the inner alpha loop after repeated DVQE-QUBO candidates fail
        to improve the alpha objective.

    - max_consecutive_outer_rejections:
        Stop the outer x-loop after repeated outer candidates fail to improve
        the original x-space objective.

    Returns
    -------
    If return_info=True:
        x_best, f_best, total_outer_iterations, info

    Else:
        x_best, f_best, total_outer_iterations
    """

    # ------------------------------------------------------------
    # Input processing
    # ------------------------------------------------------------

    A = np.asarray(A, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix.")

    n = A.shape[0]
    A = _symmetrize(A)

    if b is None:
        b = np.zeros(n, dtype=float)
    else:
        b = np.asarray(b, dtype=float)

    if b.shape != (n,):
        raise ValueError("b must have shape (n,).")

    if lb is None or ub is None:
        raise ValueError("lb and ub must be provided.")

    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    if lb.shape != (n,) or ub.shape != (n,):
        raise ValueError("lb and ub must have shape (n,).")

    if np.any(ub <= lb):
        raise ValueError("Every upper bound must be greater than lower bound.")

    var_range = ub - lb

    if x0 is None:
        x = 0.5 * (lb + ub)
    else:
        x = np.asarray(x0, dtype=float)

    if x.shape != (n,):
        raise ValueError("x0 must have shape (n,).")

    x = np.clip(x, lb, ub)

    if qpu_qubit_config is None:
        qpu_qubit_config = [n]

    rho = float(rho0)

    if rho <= 0:
        raise ValueError("rho0 must be positive.")

    if not (0 < rho_decay < 1):
        raise ValueError("rho_decay must be between 0 and 1.")

    if max_consecutive_outer_rejections is not None:
        if max_consecutive_outer_rejections <= 0:
            raise ValueError(
                "max_consecutive_outer_rejections must be positive or None."
            )

    if not (0 < alpha_step_decay < 1):
        raise ValueError("alpha_step_decay must be between 0 and 1.")

    if max_consecutive_inner_rejections is not None:
        if max_consecutive_inner_rejections <= 0:
            raise ValueError(
                "max_consecutive_inner_rejections must be positive or None."
            )

    if not (0 < alpha_warm_start_scale <= 1):
        raise ValueError("alpha_warm_start_scale must be in (0, 1].")
    # ------------------------------------------------------------
    # QUBO solver checks
    # ------------------------------------------------------------

    qubo_solver = qubo_solver.lower()

    if qubo_solver not in ["dvqe", "classical"]:
        raise ValueError("qubo_solver must be either 'dvqe' or 'classical'.")

    if classical_num_restarts <= 0:
        raise ValueError("classical_num_restarts must be positive.")

    if classical_max_sweeps <= 0:
        raise ValueError("classical_max_sweeps must be positive.")
    # ------------------------------------------------------------
    # Storage
    # ------------------------------------------------------------

    f_current = _objective(A, b, c, x)

    trajectory = [x.copy()]
    objective_history = [f_current]
    rho_history = [rho]
    accepted_outer_history = []
    outer_status_history = []

    alpha_solution_history = []
    alpha_initial_history = []
    alpha_objective_history = []
    alpha_step_history = []
    inner_status_history = []
    z_history = []

    local_qubo_history = []
    dvqe_runtime_history = []

    accepted_outer_updates = 0
    rejected_outer_updates = 0
    total_dvqe_calls = 0

    start_total = time.time()
    previous_alpha = None

    # NEW: count repeated rejected outer updates
    consecutive_outer_rejections = 0

    # ============================================================
    # OUTER LOOP IN x-SPACE
    # ============================================================

    for outer_iter in range(1, max_outer_iters + 1):

        # --------------------------------------------------------
        # Outer stopping checks
        # --------------------------------------------------------

        if rho <= rho_tol:
            outer_status_history.append("stopped: rho tolerance reached")
            break

        if grad_tol is not None:
            grad = 2.0 * A @ x + b
            if np.linalg.norm(grad, ord=np.inf) <= grad_tol:
                outer_status_history.append("stopped: gradient tolerance reached")
                break

        # --------------------------------------------------------
        # Build alpha-QUP around current x
        # --------------------------------------------------------

        A_alpha, b_alpha, c_alpha, alpha_lb, alpha_ub, var_range = _build_alpha_qup(
            A=A,
            b=b,
            c=c,
            x=x,
            lb=lb,
            ub=ub,
            rho=rho
        )

        # If trust region is numerically zero, shrink rho and continue.
        alpha_width = alpha_ub - alpha_lb

        if np.max(alpha_width) <= 1e-14:
            rho *= rho_decay
            rho_history.append(rho)
            rejected_outer_updates += 1
            consecutive_outer_rejections += 1
            outer_status_history.append("rejected: alpha region too small, rho reduced")

            if (
                max_consecutive_outer_rejections is not None
                and consecutive_outer_rejections >= max_consecutive_outer_rejections
            ):
                outer_status_history.append("stopped: consecutive outer rejections")
                break

            continue

        # --------------------------------------------------------
        # Initialize alpha for inner loop
        # --------------------------------------------------------

        if alpha0 is not None:
            alpha = np.asarray(alpha0, dtype=float)
            if alpha.shape != (n,):
                raise ValueError("alpha0 must have shape (n,).")

        elif alpha_warm_start and previous_alpha is not None:
            alpha = alpha_warm_start_scale * previous_alpha

        else:
            alpha = np.zeros(n, dtype=float)

        alpha = np.clip(alpha, alpha_lb, alpha_ub)
        alpha_initial_history.append(alpha.copy())

        if alpha_step0 is None:
            alpha_delta = alpha_step_fraction * alpha_width
        else:
            alpha_step0_arr = np.asarray(alpha_step0, dtype=float)

            if alpha_step0_arr.ndim == 0:
                alpha_delta = np.ones(n, dtype=float) * float(alpha_step0_arr)
            elif alpha_step0_arr.shape == (n,):
                alpha_delta = alpha_step0_arr.copy()
            else:
                raise ValueError("alpha_step0 must be None, scalar, or shape (n,).")

        alpha_delta = np.minimum(alpha_delta, alpha_width)
        alpha_delta = np.maximum(alpha_delta, 0.0)

        f_alpha_current = float(alpha.T @ A_alpha @ alpha + b_alpha.T @ alpha + c_alpha)

        inner_objectives = [f_alpha_current]
        inner_steps = [alpha_delta.copy()]
        inner_statuses = []

        # NEW: count repeated rejected inner DVQE-QUBO refinements
        consecutive_inner_rejections = 0

        # ========================================================
        # INNER LOOP IN alpha-SPACE
        # ========================================================

        for inner_iter in range(1, max_inner_iters + 1):

            relative_alpha_step = np.max(
                alpha_delta / np.maximum(alpha_width, 1e-12)
            )

            if relative_alpha_step <= alpha_step_tol:
                inner_statuses.append("stopped: alpha step tolerance reached")
                break

            # ----------------------------------------------------
            # Build local one-bit QUBO in alpha-space
            # ----------------------------------------------------

            Q_local, q_local, constant, y, p, d_plus, d_minus = _build_local_alpha_qubo(
                A_alpha=A_alpha,
                b_alpha=b_alpha,
                c_alpha=c_alpha,
                alpha=alpha,
                alpha_delta=alpha_delta,
                alpha_lb=alpha_lb,
                alpha_ub=alpha_ub
            )

            local_qubo_history.append({
                "outer_iter": outer_iter,
                "inner_iter": inner_iter,
                "Q": Q_local.copy(),
                "q_linear": q_local.copy(),
                "constant": constant,
                "alpha": alpha.copy(),
                "alpha_y": y.copy(),
                "alpha_p": p.copy(),
                "d_plus": d_plus.copy(),
                "d_minus": d_minus.copy(),
                "rho": rho
            })

            Q_train, q_train, scale = _scale_qubo(Q_local, q_local)

            # ----------------------------------------------------
            # Solve local alpha-QUBO using selected QUBO solver
            # ----------------------------------------------------

            qubo_start = time.time()

            try:
                qubo_seed = None
                if seed is not None:
                    qubo_seed = seed + 1000 * outer_iter + inner_iter

                if qubo_solver.lower() == "dvqe":

                    if silent:
                        buffer = io.StringIO()
                        with contextlib.redirect_stdout(buffer):
                            z_qubo, final_circuit, hist = dvqe(
                                mode=mode,
                                Q=Q_train,
                                q_linear=q_train,
                                init_type=init_type,
                                depth=depth,
                                lr=lr,
                                max_iters=max_iters,
                                qpu_qubit_config=qpu_qubit_config,
                                rel_tol=rel_tol,
                                num_shots=num_shots,
                                final_shots=final_shots,
                                warm_start_population=warm_start_population,
                                warm_start_iters=warm_start_iters,
                                warm_start_shots=warm_start_shots,
                                seed=qubo_seed,
                                verbose=False,
                                backend=backend,
                                energy_mode=energy_mode,
                                cvar_alpha=cvar_alpha
                            )
                    else:
                        z_qubo, final_circuit, hist = dvqe(
                            mode=mode,
                            Q=Q_train,
                            q_linear=q_train,
                            init_type=init_type,
                            depth=depth,
                            lr=lr,
                            max_iters=max_iters,
                            qpu_qubit_config=qpu_qubit_config,
                            rel_tol=rel_tol,
                            num_shots=num_shots,
                            final_shots=final_shots,
                            warm_start_population=warm_start_population,
                            warm_start_iters=warm_start_iters,
                            warm_start_shots=warm_start_shots,
                            seed=qubo_seed,
                            verbose=verbose,
                            backend=backend,
                            energy_mode=energy_mode,
                            cvar_alpha=cvar_alpha
                        )

                    solver_info = {
                        "solver": "dvqe",
                        "hist": hist,
                    }

                elif qubo_solver.lower() == "classical":

                    z_qubo, qubo_energy, classical_info = _solve_qubo_classical_local_search(
                        Q=Q_train,
                        q=q_train,
                        num_restarts=classical_num_restarts,
                        max_sweeps=classical_max_sweeps,
                        seed=qubo_seed,
                        initial_z=None
                    )

                    final_circuit = None
                    hist = None

                    solver_info = {
                        "solver": "classical",
                        "method": "local_search",
                        "qubo_energy": qubo_energy,
                        "classical_info": classical_info,
                    }

                else:
                    raise ValueError(
                        "qubo_solver must be either 'dvqe' or 'classical'."
                    )

                qubo_runtime = time.time() - qubo_start
                dvqe_runtime_history.append(qubo_runtime)
                total_dvqe_calls += 1

                z_qubo = np.asarray(z_qubo, dtype=int)

                if z_qubo.shape != (n,):
                    raise ValueError(
                        f"QUBO solver returned solution with shape {z_qubo.shape}, expected {(n,)}."
                    )

                z_history.append({
                    "outer_iter": outer_iter,
                    "inner_iter": inner_iter,
                    "z": z_qubo.copy(),
                    "qubo_solver": qubo_solver,
                    "solver_info": solver_info,
                })

                # ------------------------------------------------
                # Decode QUBO solution into alpha candidate
                # ------------------------------------------------

                alpha_candidate = _decode_alpha_solution(z_qubo, y, p)
                alpha_candidate = np.clip(alpha_candidate, alpha_lb, alpha_ub)

                f_alpha_candidate = float(
                    alpha_candidate.T @ A_alpha @ alpha_candidate
                    + b_alpha.T @ alpha_candidate
                    + c_alpha
                )

                improvement_alpha = f_alpha_current - f_alpha_candidate

                # ------------------------------------------------
                # Inner acceptance / rejection
                # ------------------------------------------------

                if improvement_alpha > obj_tol:
                    alpha = alpha_candidate.copy()
                    f_alpha_current = f_alpha_candidate
                    inner_status = "accepted"

                    # NEW: reset because alpha improved
                    consecutive_inner_rejections = 0

                else:
                    alpha_delta = alpha_step_decay * alpha_delta
                    inner_status = "rejected: alpha step reduced"

                    # NEW: count failed alpha refinements
                    consecutive_inner_rejections += 1

                inner_objectives.append(f_alpha_current)
                inner_steps.append(alpha_delta.copy())
                inner_statuses.append(inner_status)

                # NEW: stop inner loop after repeated failed DVQE-QUBO refinements
                if (
                    max_consecutive_inner_rejections is not None
                    and consecutive_inner_rejections >= max_consecutive_inner_rejections
                ):
                    inner_statuses.append("stopped: consecutive inner rejections")
                    break

                if verbose and not silent:
                    print(
                        f"Outer {outer_iter:3d} | Inner {inner_iter:3d} | "
                        f"f_alpha={f_alpha_current:.10f} | "
                        f"cand={f_alpha_candidate:.10f} | "
                        f"imp={improvement_alpha:.3e} | "
                        f"status={inner_status} | "
                        f"max(alpha_step/width)={relative_alpha_step:.3e} | "
                        f"inner_rej={consecutive_inner_rejections}"
                    )

            except Exception as e:
                inner_statuses.append(
                    f"failed at outer {outer_iter}, inner {inner_iter}: {str(e)}"
                )
                break

        else:
            inner_statuses.append("stopped: max inner iterations reached")

        # --------------------------------------------------------
        # End of inner loop: alpha is the best scale vector found
        # --------------------------------------------------------

        alpha_solution_history.append(alpha.copy())
        alpha_objective_history.append(inner_objectives)
        alpha_step_history.append(inner_steps)
        inner_status_history.append(inner_statuses)

        # Candidate in original x-space
        x_candidate = x + var_range * alpha
        x_candidate = np.clip(x_candidate, lb, ub)

        f_candidate = _objective(A, b, c, x_candidate)
        improvement_outer = f_current - f_candidate

        # --------------------------------------------------------
        # Outer global acceptance / rejection
        # --------------------------------------------------------

        if improvement_outer > obj_tol:
            x = x_candidate.copy()
            f_current = f_candidate
            accepted_outer = True
            accepted_outer_updates += 1
            outer_status = "accepted"

            # NEW: reset because x improved
            consecutive_outer_rejections = 0

            # Save successful alpha to warm-start the next outer iteration
            previous_alpha = alpha.copy()

        else:
            rho = rho_decay * rho
            accepted_outer = False
            rejected_outer_updates += 1
            outer_status = "rejected: rho reduced"

            # NEW: count failed outer update
            consecutive_outer_rejections += 1

            # Optional: weaken the previous alpha after failed outer update
            if previous_alpha is not None:
                previous_alpha = alpha_warm_start_scale * previous_alpha

        trajectory.append(x.copy())
        objective_history.append(f_current)
        rho_history.append(rho)
        accepted_outer_history.append(accepted_outer)
        outer_status_history.append(outer_status)

        if verbose and not silent:
            print(
                f"OUTER {outer_iter:3d} | "
                f"f={f_current:.10f} | "
                f"candidate={f_candidate:.10f} | "
                f"improvement={improvement_outer:.3e} | "
                f"accepted={accepted_outer} | "
                f"rho={rho:.3e} | "
                f"dvqe_calls={total_dvqe_calls} | "
                f"outer_rej={consecutive_outer_rejections}"
            )

        # NEW: stop outer loop after repeated failed outer updates
        if (
            max_consecutive_outer_rejections is not None
            and consecutive_outer_rejections >= max_consecutive_outer_rejections
        ):
            outer_status_history.append("stopped: consecutive outer rejections")
            break

    else:
        outer_status_history.append("stopped: max outer iterations reached")

    # ------------------------------------------------------------
    # Return result
    # ------------------------------------------------------------

    total_runtime = time.time() - start_total
    total_outer_iterations = len(accepted_outer_history)

    info = {
        "trajectory": trajectory,
        "objective_history": objective_history,
        "rho_history": rho_history,
        "accepted_outer_history": accepted_outer_history,
        "outer_status_history": outer_status_history,

        "alpha_initial_history": alpha_initial_history,
        "alpha_solution_history": alpha_solution_history,
        "alpha_objective_history": alpha_objective_history,
        "alpha_step_history": alpha_step_history,
        "inner_status_history": inner_status_history,

        "z_history": z_history,
        "local_qubo_history": local_qubo_history,
        "dvqe_runtime_history": dvqe_runtime_history,

        "accepted_outer_updates": accepted_outer_updates,
        "rejected_outer_updates": rejected_outer_updates,
        "total_outer_iterations": total_outer_iterations,
        "total_dvqe_calls": total_dvqe_calls,
        "total_runtime_sec": total_runtime,

        "final_rho": rho,
        "qpu_qubit_config": qpu_qubit_config,
        "mode": mode,
        "silent": silent,

        # QUBO solver settings
        "qubo_solver": qubo_solver,
        "classical_num_restarts": classical_num_restarts,
        "classical_max_sweeps": classical_max_sweeps,

        # NEW saved settings
        "alpha_warm_start": alpha_warm_start,
        "alpha_warm_start_scale": alpha_warm_start_scale,
        "max_consecutive_inner_rejections": max_consecutive_inner_rejections,
        "max_consecutive_outer_rejections": max_consecutive_outer_rejections,
    }

    if return_info:
        return x, f_current, total_outer_iterations, info

    return x, f_current, total_outer_iterations



# ============================================================
# PHR FIXED-REGION HELPERS
# ============================================================

def _phr_augmented_objective(
    A,
    b,
    c,
    x,
    G,
    r,
    H,
    h,
    lambda_eq,
    lambda_ineq,
    mu
):
    """
    Evaluate the Powell-Hestenes-Rockafellar augmented Lagrangian.

    Original QP:
        min f(x) = x^T A x + b^T x + c

        s.t. Gx = r
             Hx <= h

    Let:
        e(x) = Gx - r
        g(x) = Hx - h

    The PHR augmented Lagrangian is:

        L_mu(x, lambda_eq, lambda_ineq)
        = f(x)
          + lambda_eq^T e(x)
          + mu/2 ||e(x)||^2
          + 1/(2 mu) (
                ||max(0, lambda_ineq + mu g(x))||^2
                - ||lambda_ineq||^2
            )

    The maximum is componentwise.
    """
    value = _objective(A, b, c, x)

    if G.shape[0] > 0:
        eq_residual = G @ x - r
        value += float(lambda_eq.T @ eq_residual)
        value += 0.5 * mu * float(eq_residual.T @ eq_residual)

    if H.shape[0] > 0:
        ineq_residual = H @ x - h
        shifted = lambda_ineq + mu * ineq_residual
        positive_shifted = np.maximum(shifted, 0.0)

        value += (
            float(positive_shifted.T @ positive_shifted)
            - float(lambda_ineq.T @ lambda_ineq)
        ) / (2.0 * mu)

    return float(value)


def _build_phr_fixed_region_qp(
    A,
    b,
    c,
    G,
    r,
    H,
    h,
    lambda_eq,
    lambda_ineq,
    mu,
    region_mask
):
    """
    Build one ordinary QP corresponding to a fixed PHR region.

    For a selected PHR region P:

        P = { i : lambda_i + mu (H_i x - h_i) > 0 }

    the inequality contribution for i in P is:

        lambda_i (H_i x - h_i)
        + mu/2 (H_i x - h_i)^2

    while a row outside P contributes only a constant.

    Therefore, for a fixed region, the PHR AL is exactly a quadratic
    function of x. DQUP can solve this n-variable bounded QP directly.

    The returned constant includes the PHR term:
        - ||lambda_ineq||^2 / (2 mu)
    so the returned QP objective equals the fixed-region PHR expression.
    """
    n = A.shape[0]

    region_mask = np.asarray(region_mask, dtype=bool)

    if region_mask.shape != (H.shape[0],):
        raise ValueError("region_mask must have shape (number of inequalities,).")

    # Equality contribution:
    #
    # lambda_eq^T(Gx-r) + mu/2 ||Gx-r||^2
    #
    A_aug = A.copy()
    b_aug = b.copy()
    c_aug = float(c)

    if G.shape[0] > 0:
        A_aug += 0.5 * mu * (G.T @ G)
        b_aug += G.T @ lambda_eq - mu * (G.T @ r)

        c_aug += (
            -float(lambda_eq.T @ r)
            + 0.5 * mu * float(r.T @ r)
        )

    # PHR inequality contribution for the currently selected region.
    active_indices = np.where(region_mask)[0]

    if active_indices.size > 0:
        H_region = H[active_indices, :]
        h_region = h[active_indices]
        lambda_region = lambda_ineq[active_indices]

        A_aug += 0.5 * mu * (H_region.T @ H_region)
        b_aug += (
            H_region.T @ lambda_region
            - mu * (H_region.T @ h_region)
        )

        c_aug += (
            -float(lambda_region.T @ h_region)
            + 0.5 * mu * float(h_region.T @ h_region)
        )

    # This constant does not affect the x minimizer, but including it
    # makes the fixed-region objective equal to the PHR expression.
    if H.shape[0] > 0:
        c_aug -= 0.5 * float(lambda_ineq.T @ lambda_ineq) / mu

    return (
        _symmetrize(A_aug),
        b_aug,
        float(c_aug),
        active_indices
    )


# ============================================================
# MAIN SOLVER: FIXED-REGION PHR DQP
# ============================================================

def dqp(
    A,
    b=None,
    c=0.0,

    # Equality constraints: Gx = r
    G=None,
    r=None,

    # Inequality constraints: Hx <= h
    H=None,
    h=None,

    x0=None,
    lb=None,
    ub=None,

    # Augmented-Lagrangian settings
    lambda0=None,
    mu0=100.0,
    mu_min=1e-6,
    mu_max=1e8,
    constraint_tol=1e-4,
    stationarity_tol=None,
    max_al_iters=20,
    min_al_iters=3,

    # Fixed PHR-region settings
    phr_region_tol=1e-12,
    max_region_iters=8,

    # Mu update settings
    update_mu=True,
    mu_update_rule="primal_dual_balance",
    balance_factor=10.0,
    mu_increase=2.0,
    mu_decrease=2.0,
    allow_mu_decrease=True,
    primal_stall_ratio=0.90,

    # Printing controls
    verbose=False,

    # Output settings
    return_info=True,

    # Passed into dqup()
    **dqup_kwargs
):
    """
    Fixed-region Powell-Hestenes-Rockafellar (PHR) augmented-
    Lagrangian DQP solver for bounded QPs with linear equalities
    and inequalities.

    Solves approximately:

        min_x  x^T A x + b^T x + c

        s.t.   Gx = r
               Hx <= h
               lb <= x <= ub

    No slack variables are introduced. Every call to DQUP has the
    original dimension n.

    PHR inequality formulation:
    ---------------------------
    Let:

        g(x) = Hx - h <= 0

    and let lambda_ineq >= 0. The PHR AL term is:

        1/(2 mu) [
            ||max(0, lambda_ineq + mu g(x))||^2
            - ||lambda_ineq||^2
        ]

    This function is piecewise quadratic. At a fixed region P:

        P = {i : lambda_i + mu g_i(x) > 0},

    it becomes one ordinary quadratic function of x. This solver
    repeatedly:

        1) chooses P from the current x,
        2) builds the corresponding fixed-region QP,
        3) solves that QP with DQUP,
        4) recomputes P,
        5) repeats until P is stable or max_region_iters is reached.

    Multiplier updates:
    -------------------
    Equality multipliers are unrestricted:

        lambda_eq <- lambda_eq + mu (Gx-r)

    Inequality multipliers are projected:

        lambda_ineq <- max(0, lambda_ineq + mu (Hx-h))

    Notes:
    ------
    A fixed-region DQUP subproblem is a QP, but global convergence
    still depends on sufficiently accurate DQUP solves, sensible mu
    updates, and successful PHR-region stabilization.
    """

    # ------------------------------------------------------------
    # Input processing
    # ------------------------------------------------------------

    A = np.asarray(A, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix.")

    n = A.shape[0]
    A = _symmetrize(A)

    if b is None:
        b = np.zeros(n, dtype=float)
    else:
        b = np.asarray(b, dtype=float)

    if b.shape != (n,):
        raise ValueError("b must have shape (n,).")

    if lb is None or ub is None:
        raise ValueError("lb and ub must be provided.")

    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    if lb.shape != (n,) or ub.shape != (n,):
        raise ValueError("lb and ub must have shape (n,).")

    if np.any(ub <= lb):
        raise ValueError("Every upper bound must be greater than lower bound.")

    # ------------------------------------------------------------
    # Equality constraints Gx = r
    # ------------------------------------------------------------

    if (G is None) != (r is None):
        raise ValueError("G and r must be both provided or both None.")

    has_eq = G is not None and r is not None

    if has_eq:
        G = np.asarray(G, dtype=float)
        r = np.asarray(r, dtype=float)

        if G.ndim != 2 or G.shape[1] != n:
            raise ValueError("G must have shape (m, n).")

        m = G.shape[0]

        if r.shape != (m,):
            raise ValueError("r must have shape (m,).")

    else:
        G = np.zeros((0, n), dtype=float)
        r = np.zeros(0, dtype=float)
        m = 0

    # ------------------------------------------------------------
    # Inequality constraints Hx <= h
    # ------------------------------------------------------------

    if (H is None) != (h is None):
        raise ValueError("H and h must be both provided or both None.")

    has_ineq = H is not None and h is not None

    if has_ineq:
        H = np.asarray(H, dtype=float)
        h = np.asarray(h, dtype=float)

        if H.ndim != 2 or H.shape[1] != n:
            raise ValueError("H must have shape (p, n).")

        p = H.shape[0]

        if h.shape != (p,):
            raise ValueError("h must have shape (p,).")

    else:
        H = np.zeros((0, n), dtype=float)
        h = np.zeros(0, dtype=float)
        p = 0

    if m == 0 and p == 0:
        raise ValueError(
            "At least one of Gx=r or Hx<=h must be provided."
        )

    # ------------------------------------------------------------
    # Initial x
    # ------------------------------------------------------------

    if x0 is None:
        x = 0.5 * (lb + ub)
    else:
        x = np.asarray(x0, dtype=float)

    if x.shape != (n,):
        raise ValueError("x0 must have shape (n,).")

    x = np.clip(x, lb, ub)

    # ------------------------------------------------------------
    # Settings validation
    # ------------------------------------------------------------

    mu = float(mu0)

    if mu <= 0:
        raise ValueError("mu0 must be positive.")

    if mu_min <= 0:
        raise ValueError("mu_min must be positive.")

    if mu_max < mu_min:
        raise ValueError("mu_max must be greater than or equal to mu_min.")

    if max_al_iters <= 0:
        raise ValueError("max_al_iters must be positive.")

    if min_al_iters < 1:
        raise ValueError("min_al_iters must be at least 1.")

    if max_region_iters <= 0:
        raise ValueError("max_region_iters must be positive.")

    phr_region_tol = float(phr_region_tol)

    if phr_region_tol < 0:
        raise ValueError("phr_region_tol must be nonnegative.")

    if balance_factor <= 1:
        raise ValueError("balance_factor must be greater than 1.")

    if mu_increase <= 1:
        raise ValueError("mu_increase must be greater than 1.")

    if mu_decrease <= 1:
        raise ValueError("mu_decrease must be greater than 1.")

    if not (0.0 < primal_stall_ratio <= 1.0):
        raise ValueError("primal_stall_ratio must be in (0, 1].")

    # Do not allow duplicate DQUP arguments.
    for key in ["A", "b", "c", "x0", "lb", "ub", "return_info"]:
        dqup_kwargs.pop(key, None)

    # ------------------------------------------------------------
    # Multiplier initialization
    # ------------------------------------------------------------

    total_constraints = m + p

    if lambda0 is None:
        lambda_eq = np.zeros(m, dtype=float)
        lambda_ineq = np.zeros(p, dtype=float)

    else:
        lambda0 = np.asarray(lambda0, dtype=float)

        if lambda0.shape != (total_constraints,):
            raise ValueError("lambda0 must have shape (m+p,).")

        lambda_eq = lambda0[:m].copy()
        lambda_ineq = np.maximum(0.0, lambda0[m:m + p])

    # ------------------------------------------------------------
    # Storage
    # ------------------------------------------------------------

    start_total = time.time()

    x_history = [x.copy()]

    lambda_eq_history = [lambda_eq.copy()]
    lambda_ineq_history = [lambda_ineq.copy()]
    lambda_history = [np.concatenate([lambda_eq, lambda_ineq])]

    mu_history = [mu]

    original_objective_history = [_objective(A, b, c, x)]
    phr_augmented_objective_history = [
        _phr_augmented_objective(
            A, b, c, x, G, r, H, h, lambda_eq, lambda_ineq, mu
        )
    ]

    eq_residual_history = []
    ineq_residual_history = []
    ineq_violation_history = []

    eq_residual_norm_inf_history = []
    ineq_residual_norm_inf_history = []
    ineq_violation_inf_history = []
    primal_residual_history = []
    dual_residual_history = []
    stationarity_norm_inf_history = []
    complementarity_norm_inf_history = []

    # PHR fixed-region histories.
    phr_region_mask_history = []
    phr_region_indices_history = []
    phr_region_count_history = []
    phr_region_stable_history = []
    phr_region_iterations_history = []
    phr_region_info_history = []

    # Compatibility aliases for previous runner code.
    active_set_history = []
    active_indices_history = []
    num_active_ineq_history = []

    dqup_info_history = []
    dqup_iterations_history = []
    dqup_runtime_history = []

    mu_update_history = []
    status_history = []

    best_feasible_x = None
    best_feasible_f = np.inf

    previous_primal_residual = np.inf

    # ============================================================
    # OUTER PHR AUGMENTED-LAGRANGIAN LOOP
    # ============================================================

    for al_iter in range(1, max_al_iters + 1):

        x_region = x.copy()
        region_stable = (p == 0)
        final_region_mask = np.zeros(p, dtype=bool)
        final_region_indices = np.zeros(0, dtype=int)

        final_dqup_info = None
        final_dqup_outer_iters = 0
        total_region_runtime = 0.0
        region_records = []

        # ========================================================
        # INNER FIXED-REGION LOOP
        # ========================================================

        for region_iter in range(1, max_region_iters + 1):

            if p > 0:
                g_region = H @ x_region - h
                shifted_region = lambda_ineq + mu * g_region

                region_mask = shifted_region > phr_region_tol
            else:
                g_region = np.zeros(0, dtype=float)
                shifted_region = np.zeros(0, dtype=float)
                region_mask = np.zeros(0, dtype=bool)

            (
                A_region,
                b_region,
                c_region,
                region_indices
            ) = _build_phr_fixed_region_qp(
                A=A,
                b=b,
                c=c,
                G=G,
                r=r,
                H=H,
                h=h,
                lambda_eq=lambda_eq,
                lambda_ineq=lambda_ineq,
                mu=mu,
                region_mask=region_mask
            )

            dqup_start = time.time()

            x_candidate, fixed_region_objective, dqup_outer_iters, dqup_info = dqup(
                A=A_region,
                b=b_region,
                c=c_region,
                x0=x_region,
                lb=lb,
                ub=ub,
                return_info=True,
                **dqup_kwargs
            )

            dqup_runtime = time.time() - dqup_start
            total_region_runtime += dqup_runtime

            x_candidate = np.clip(x_candidate, lb, ub)

            if p > 0:
                g_candidate = H @ x_candidate - h
                shifted_candidate = lambda_ineq + mu * g_candidate
                candidate_region_mask = shifted_candidate > phr_region_tol
            else:
                g_candidate = np.zeros(0, dtype=float)
                shifted_candidate = np.zeros(0, dtype=float)
                candidate_region_mask = np.zeros(0, dtype=bool)

            region_records.append({
                "region_iteration": region_iter,
                "region_mask_before_solve": region_mask.copy(),
                "region_indices_before_solve": region_indices.copy(),
                "shifted_before_solve": shifted_region.copy(),
                "x_before_solve": x_region.copy(),
                "x_after_solve": x_candidate.copy(),
                "ineq_residual_after_solve": g_candidate.copy(),
                "shifted_after_solve": shifted_candidate.copy(),
                "region_mask_after_solve": candidate_region_mask.copy(),
                "region_stable": bool(
                    np.array_equal(region_mask, candidate_region_mask)
                ),
                "fixed_region_objective": float(fixed_region_objective),
                "dqup_info": dqup_info,
                "dqup_outer_iterations": int(dqup_outer_iters),
                "dqup_runtime_sec": float(dqup_runtime),
            })

            x_region = x_candidate.copy()
            final_region_mask = region_mask.copy()
            final_region_indices = region_indices.copy()
            final_dqup_info = dqup_info
            final_dqup_outer_iters = int(dqup_outer_iters)

            if np.array_equal(region_mask, candidate_region_mask):
                region_stable = True
                break

        # Accept the final point generated by the fixed-region loop.
        x_before_al = x.copy()
        x = x_region.copy()

        # Recompute the PHR region at the accepted point.
        if p > 0:
            ineq_residual = H @ x - h
            ineq_violation = np.maximum(ineq_residual, 0.0)

            shifted_final_before_update = (
                lambda_ineq + mu * ineq_residual
            )

            final_region_mask_at_x = (
                shifted_final_before_update > phr_region_tol
            )

            final_region_indices_at_x = np.where(
                final_region_mask_at_x
            )[0]

        else:
            ineq_residual = np.zeros(0, dtype=float)
            ineq_violation = np.zeros(0, dtype=float)
            shifted_final_before_update = np.zeros(0, dtype=float)
            final_region_mask_at_x = np.zeros(0, dtype=bool)
            final_region_indices_at_x = np.zeros(0, dtype=int)

        if m > 0:
            eq_residual = G @ x - r
        else:
            eq_residual = np.zeros(0, dtype=float)

        f_original = _objective(A, b, c, x)

        phr_augmented_objective = _phr_augmented_objective(
            A, b, c, x, G, r, H, h, lambda_eq, lambda_ineq, mu
        )

        eq_residual_norm_inf = (
            float(np.linalg.norm(eq_residual, ord=np.inf))
            if m > 0 else 0.0
        )

        ineq_residual_norm_inf = (
            float(np.linalg.norm(ineq_residual, ord=np.inf))
            if p > 0 else 0.0
        )

        ineq_violation_inf = (
            float(np.linalg.norm(ineq_violation, ord=np.inf))
            if p > 0 else 0.0
        )

        primal_residual = max(
            eq_residual_norm_inf,
            ineq_violation_inf
        )

        # --------------------------------------------------------
        # Multiplier updates
        # --------------------------------------------------------

        if m > 0:
            lambda_eq_new = lambda_eq + mu * eq_residual
        else:
            lambda_eq_new = np.zeros(0, dtype=float)

        if p > 0:
            lambda_ineq_new = np.maximum(
                0.0,
                lambda_ineq + mu * ineq_residual
            )
        else:
            lambda_ineq_new = np.zeros(0, dtype=float)

        # --------------------------------------------------------
        # KKT-related diagnostics
        # --------------------------------------------------------

        grad_x = 2.0 * A @ x + b

        stationarity = grad_x.copy()

        if m > 0:
            stationarity += G.T @ lambda_eq_new

        if p > 0:
            stationarity += H.T @ lambda_ineq_new

        stationarity_norm_inf = float(
            np.linalg.norm(stationarity, ord=np.inf)
        )

        if p > 0:
            complementarity_norm_inf = float(
                np.linalg.norm(
                    lambda_ineq_new * ineq_residual,
                    ord=np.inf
                )
            )
        else:
            complementarity_norm_inf = 0.0

        # This is an AL-change diagnostic, not an exact KKT dual
        # residual. It uses the final fixed PHR region.
        if m == 0 and final_region_indices_at_x.size == 0:
            dual_residual = np.inf
        else:
            dx = x - x_before_al
            dual_vec = np.zeros(n, dtype=float)

            if m > 0:
                dual_vec += mu * (G.T @ (G @ dx))

            if final_region_indices_at_x.size > 0:
                H_final_region = H[final_region_indices_at_x, :]
                dual_vec += mu * (
                    H_final_region.T @ (H_final_region @ dx)
                )

            dual_residual = float(
                np.linalg.norm(dual_vec, ord=np.inf)
            )

        # --------------------------------------------------------
        # Convergence test
        # --------------------------------------------------------

        eq_satisfied = eq_residual_norm_inf <= constraint_tol
        ineq_satisfied = ineq_violation_inf <= constraint_tol
        constraint_satisfied = eq_satisfied and ineq_satisfied

        if stationarity_tol is None:
            stationarity_satisfied = True
        else:
            stationarity_satisfied = (
                stationarity_norm_inf <= stationarity_tol
            )

        if constraint_satisfied and f_original < best_feasible_f:
            best_feasible_x = x.copy()
            best_feasible_f = f_original

        # --------------------------------------------------------
        # Store histories
        # --------------------------------------------------------

        x_history.append(x.copy())

        # Histories store the multipliers that produced this AL step.
        lambda_eq_history.append(lambda_eq.copy())
        lambda_ineq_history.append(lambda_ineq.copy())
        lambda_history.append(np.concatenate([lambda_eq, lambda_ineq]))

        mu_history.append(mu)

        original_objective_history.append(f_original)
        phr_augmented_objective_history.append(phr_augmented_objective)

        eq_residual_history.append(eq_residual.copy())
        ineq_residual_history.append(ineq_residual.copy())
        ineq_violation_history.append(ineq_violation.copy())

        eq_residual_norm_inf_history.append(eq_residual_norm_inf)
        ineq_residual_norm_inf_history.append(ineq_residual_norm_inf)
        ineq_violation_inf_history.append(ineq_violation_inf)

        primal_residual_history.append(primal_residual)
        dual_residual_history.append(dual_residual)
        stationarity_norm_inf_history.append(stationarity_norm_inf)
        complementarity_norm_inf_history.append(complementarity_norm_inf)

        phr_region_mask_history.append(final_region_mask_at_x.copy())
        phr_region_indices_history.append(
            final_region_indices_at_x.copy()
        )
        phr_region_count_history.append(
            int(final_region_indices_at_x.size)
        )
        phr_region_stable_history.append(bool(region_stable))
        phr_region_iterations_history.append(len(region_records))
        phr_region_info_history.append(region_records)

        # Compatibility aliases.
        active_set_history.append(final_region_mask_at_x.copy())
        active_indices_history.append(
            final_region_indices_at_x.copy()
        )
        num_active_ineq_history.append(
            int(final_region_indices_at_x.size)
        )

        dqup_info_history.append(final_dqup_info)
        dqup_iterations_history.append(final_dqup_outer_iters)
        dqup_runtime_history.append(total_region_runtime)

        # --------------------------------------------------------
        # Stop if converged
        # --------------------------------------------------------

        if (
            al_iter >= min_al_iters
            and constraint_satisfied
            and stationarity_satisfied
        ):
            status_history.append("converged")

            mu_update_history.append({
                "iteration": al_iter,
                "mu_before": mu,
                "mu_after": mu,
                "status": "not updated: converged",
                "primal_residual": primal_residual,
                "dual_residual": dual_residual,
                "eq_residual_inf": eq_residual_norm_inf,
                "ineq_violation_inf": ineq_violation_inf,
                "phr_region_count": int(final_region_indices_at_x.size),
                "phr_region_indices": final_region_indices_at_x.copy(),
                "region_stable": bool(region_stable),
                "region_iterations": len(region_records),
            })

            lambda_eq = lambda_eq_new.copy()
            lambda_ineq = lambda_ineq_new.copy()
            break

        # --------------------------------------------------------
        # Accept multiplier update
        # --------------------------------------------------------

        lambda_eq = lambda_eq_new.copy()
        lambda_ineq = lambda_ineq_new.copy()

        # --------------------------------------------------------
        # Mu update
        # --------------------------------------------------------

        mu_before = mu
        mu_update_status = "not updated"

        stalled_primal = (
            np.isfinite(previous_primal_residual)
            and primal_residual > constraint_tol
            and primal_residual >= (
                primal_stall_ratio * previous_primal_residual
            )
        )

        if update_mu and mu_update_rule == "primal_dual_balance":

            if stalled_primal:
                mu = min(mu_increase * mu, mu_max)
                mu_update_status = "increased: primal residual stalled"

            elif np.isfinite(dual_residual):

                if primal_residual > balance_factor * dual_residual:
                    mu = min(mu_increase * mu, mu_max)
                    mu_update_status = "increased: primal dominates"

                elif (
                    allow_mu_decrease
                    and dual_residual > balance_factor * primal_residual
                ):
                    mu = max(mu / mu_decrease, mu_min)
                    mu_update_status = "decreased: dual dominates"

                else:
                    mu_update_status = "unchanged: balanced"

            else:
                mu_update_status = "unchanged: no AL penalty rows"

        elif not update_mu:
            mu_update_status = "unchanged: update_mu=False"

        else:
            raise ValueError(
                f"Unknown mu_update_rule: {mu_update_rule}. "
                "Supported: 'primal_dual_balance'."
            )

        mu_update_history.append({
            "iteration": al_iter,
            "mu_before": mu_before,
            "mu_after": mu,
            "status": mu_update_status,
            "primal_residual": primal_residual,
            "dual_residual": dual_residual,
            "eq_residual_inf": eq_residual_norm_inf,
            "ineq_residual_inf": ineq_residual_norm_inf,
            "ineq_violation_inf": ineq_violation_inf,
            "phr_region_count": int(final_region_indices_at_x.size),
            "phr_region_indices": final_region_indices_at_x.copy(),
            "region_stable": bool(region_stable),
            "region_iterations": len(region_records),
        })

        previous_primal_residual = primal_residual
        status_history.append("continued")

        if verbose:
            print(
                f"AL {al_iter:3d} | "
                f"f={f_original:.10f} | "
                f"phr={phr_augmented_objective:.10f} | "
                f"eq={eq_residual_norm_inf:.3e} | "
                f"ineq={ineq_violation_inf:.3e} | "
                f"region={final_region_indices_at_x.size}/{p} | "
                f"region_iters={len(region_records)} | "
                f"stable={region_stable} | "
                f"primal={primal_residual:.3e} | "
                f"dual={dual_residual:.3e} | "
                f"stat={stationarity_norm_inf:.3e} | "
                f"comp={complementarity_norm_inf:.3e} | "
                f"mu={mu_before:.3e}->{mu:.3e} | "
                f"mu_status={mu_update_status}"
            )

    else:
        status_history.append(
            "stopped: max augmented-Lagrangian iterations reached"
        )

    # ============================================================
    # FINAL OUTPUT
    # ============================================================

    total_runtime = time.time() - start_total
    total_al_iterations = len(eq_residual_history)

    f_final_original = _objective(A, b, c, x)

    if m > 0:
        final_eq_residual = G @ x - r
    else:
        final_eq_residual = np.zeros(0, dtype=float)

    if p > 0:
        final_ineq_residual = H @ x - h
        final_ineq_violation = np.maximum(
            final_ineq_residual,
            0.0
        )

        final_shifted = (
            lambda_ineq + mu * final_ineq_residual
        )

        final_phr_region_mask = (
            final_shifted > phr_region_tol
        )

        final_phr_region_indices = np.where(
            final_phr_region_mask
        )[0]

    else:
        final_ineq_residual = np.zeros(0, dtype=float)
        final_ineq_violation = np.zeros(0, dtype=float)
        final_shifted = np.zeros(0, dtype=float)
        final_phr_region_mask = np.zeros(0, dtype=bool)
        final_phr_region_indices = np.zeros(0, dtype=int)

    final_eq_residual_norm_inf = (
        float(np.linalg.norm(final_eq_residual, ord=np.inf))
        if m > 0 else 0.0
    )

    final_ineq_residual_norm_inf = (
        float(np.linalg.norm(final_ineq_residual, ord=np.inf))
        if p > 0 else 0.0
    )

    final_ineq_violation_inf = (
        float(np.linalg.norm(final_ineq_violation, ord=np.inf))
        if p > 0 else 0.0
    )

    final_residual_norm_inf = max(
        final_eq_residual_norm_inf,
        final_ineq_violation_inf
    )

    final_lambda = np.concatenate([lambda_eq, lambda_ineq])

    final_stationarity = 2.0 * A @ x + b

    if m > 0:
        final_stationarity += G.T @ lambda_eq

    if p > 0:
        final_stationarity += H.T @ lambda_ineq

    final_stationarity_norm_inf = float(
        np.linalg.norm(final_stationarity, ord=np.inf)
    )

    if p > 0:
        final_complementarity_norm_inf = float(
            np.linalg.norm(
                lambda_ineq * final_ineq_residual,
                ord=np.inf
            )
        )
    else:
        final_complementarity_norm_inf = 0.0

    qubo_solver_used = dqup_kwargs.get("qubo_solver", "dvqe")
    classical_num_restarts_used = dqup_kwargs.get(
        "classical_num_restarts",
        None
    )
    classical_max_sweeps_used = dqup_kwargs.get(
        "classical_max_sweeps",
        None
    )

    info = {
        "x_history": x_history,

        "lambda_history": lambda_history,
        "lambda_eq_history": lambda_eq_history,
        "lambda_ineq_history": lambda_ineq_history,
        "mu_history": mu_history,

        "original_objective_history": original_objective_history,
        "phr_augmented_objective_history": phr_augmented_objective_history,

        # Alias retained for code that used the old name.
        "augmented_objective_history": phr_augmented_objective_history,

        "eq_residual_history": eq_residual_history,
        "ineq_residual_history": ineq_residual_history,
        "ineq_violation_history": ineq_violation_history,

        "eq_residual_norm_inf_history": eq_residual_norm_inf_history,
        "ineq_residual_norm_inf_history": ineq_residual_norm_inf_history,
        "ineq_violation_inf_history": ineq_violation_inf_history,

        "primal_residual_history": primal_residual_history,
        "dual_residual_history": dual_residual_history,
        "stationarity_norm_inf_history": stationarity_norm_inf_history,
        "complementarity_norm_inf_history": complementarity_norm_inf_history,

        "phr_region_mask_history": phr_region_mask_history,
        "phr_region_indices_history": phr_region_indices_history,
        "phr_region_count_history": phr_region_count_history,
        "phr_region_stable_history": phr_region_stable_history,
        "phr_region_iterations_history": phr_region_iterations_history,
        "phr_region_info_history": phr_region_info_history,

        # Compatibility aliases for the prior runner.
        "active_set_history": active_set_history,
        "active_indices_history": active_indices_history,
        "num_active_ineq_history": num_active_ineq_history,

        "dqup_info_history": dqup_info_history,
        "dqup_iterations_history": dqup_iterations_history,
        "dqup_runtime_history": dqup_runtime_history,

        "mu_update_history": mu_update_history,
        "status_history": status_history,

        "final_x": x.copy(),
        "final_original_objective": f_final_original,

        "final_eq_residual": final_eq_residual.copy(),
        "final_ineq_residual": final_ineq_residual.copy(),
        "final_ineq_violation": final_ineq_violation.copy(),

        "final_eq_residual_norm_inf": final_eq_residual_norm_inf,
        "final_ineq_residual_norm_inf": final_ineq_residual_norm_inf,
        "final_ineq_violation_inf": final_ineq_violation_inf,
        "final_residual_norm_inf": final_residual_norm_inf,

        "final_lambda": final_lambda.copy(),
        "final_lambda_eq": lambda_eq.copy(),
        "final_lambda_ineq": lambda_ineq.copy(),
        "final_mu": mu,

        "final_phr_shifted_residual": final_shifted.copy(),
        "final_phr_region_mask": final_phr_region_mask.copy(),
        "final_phr_region_indices": final_phr_region_indices.copy(),

        "final_stationarity_norm_inf": final_stationarity_norm_inf,
        "final_complementarity_norm_inf": (
            final_complementarity_norm_inf
        ),

        "best_feasible_x": best_feasible_x,
        "best_feasible_objective": best_feasible_f,

        "has_eq": has_eq,
        "has_ineq": has_ineq,
        "num_equalities": m,
        "num_inequalities": p,

        "combined_dimension": n,
        "used_slack_variables": False,

        "x_lb": lb.copy(),
        "x_ub": ub.copy(),

        "constraint_tol": constraint_tol,
        "stationarity_tol": stationarity_tol,

        "phr_region_tol": phr_region_tol,
        "max_region_iters": max_region_iters,

        "min_al_iters": min_al_iters,
        "max_al_iters": max_al_iters,

        "update_mu": update_mu,
        "mu_update_rule": mu_update_rule,
        "balance_factor": balance_factor,
        "mu_increase": mu_increase,
        "mu_decrease": mu_decrease,
        "allow_mu_decrease": allow_mu_decrease,
        "primal_stall_ratio": primal_stall_ratio,
        "mu_min": mu_min,
        "mu_max": mu_max,

        "total_al_iterations": total_al_iterations,
        "total_runtime_sec": total_runtime,

        "qubo_solver": qubo_solver_used,
        "classical_num_restarts": classical_num_restarts_used,
        "classical_max_sweeps": classical_max_sweeps_used,
    }

    if return_info:
        return x, f_final_original, total_al_iterations, info

    return x, f_final_original, total_al_iterations


# ============================================================
# SCALED DQP WRAPPER FOR BADLY SCALED QPs
# ============================================================

def _normalize_constraint_rows(M, rhs):
    """
    Normalize rows of M y <= rhs or M y = rhs.
    """
    M = np.asarray(M, dtype=float)
    rhs = np.asarray(rhs, dtype=float)

    if M.shape[0] == 0:
        return M, rhs, np.ones(0, dtype=float)

    row_norm = np.maximum(np.linalg.norm(M, axis=1), 1e-12)

    M_scaled = M / row_norm[:, None]
    rhs_scaled = rhs / row_norm

    return M_scaled, rhs_scaled, row_norm


def _add_small_regularization(A, b, c, y_ref=None, eps_reg=1e-8):
    """
    Add eps_reg * ||y - y_ref||^2 to improve weak-cost variables.
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    n = A.shape[0]

    if y_ref is None:
        y_ref = np.zeros(n, dtype=float)
    else:
        y_ref = np.asarray(y_ref, dtype=float)

    A_reg = A + eps_reg * np.eye(n)
    b_reg = b - 2.0 * eps_reg * y_ref
    c_reg = float(c + eps_reg * float(y_ref.T @ y_ref))

    return A_reg, b_reg, c_reg


def scaled_dqp(
    A,
    b=None,
    c=0.0,

    G=None,
    r=None,

    H=None,
    h=None,

    x0=None,
    lb=None,
    ub=None,

    # Scaling controls
    normalize_constraints=True,
    scale_objective=True,
    regularize_weak_cost=True,
    eps_reg=1e-8,

    return_info=True,

    **dqp_kwargs
):
    """
    Scaled wrapper around dqp().

    It solves the QP in normalized variables:

        x = x_center + x_scale * y

    where:

        -1 <= y <= 1

    This improves robustness for badly scaled QPs.
    """

    A = np.asarray(A, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix.")

    n = A.shape[0]

    if b is None:
        b = np.zeros(n, dtype=float)
    else:
        b = np.asarray(b, dtype=float)

    if lb is None or ub is None:
        raise ValueError("lb and ub must be provided.")

    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)

    if G is None:
        G = np.zeros((0, n), dtype=float)
        r = np.zeros(0, dtype=float)
    else:
        G = np.asarray(G, dtype=float)
        r = np.asarray(r, dtype=float)

    if H is None:
        H = np.zeros((0, n), dtype=float)
        h = np.zeros(0, dtype=float)
    else:
        H = np.asarray(H, dtype=float)
        h = np.asarray(h, dtype=float)

    # ------------------------------------------------------------
    # 1) Variable scaling: x = x_center + x_scale * y
    # ------------------------------------------------------------

    x_center = 0.5 * (lb + ub)
    x_scale = 0.5 * (ub - lb)

    x_scale = np.maximum(x_scale, 1e-12)

    S = np.diag(x_scale)

    y_lb = -np.ones(n, dtype=float)
    y_ub = np.ones(n, dtype=float)

    if x0 is None:
        y0 = np.zeros(n, dtype=float)
    else:
        x0 = np.asarray(x0, dtype=float)
        x0 = np.clip(x0, lb, ub)
        y0 = (x0 - x_center) / x_scale
        y0 = np.clip(y0, y_lb, y_ub)

    # ------------------------------------------------------------
    # 2) Transform objective to y-space
    # ------------------------------------------------------------

    A_y = S.T @ A @ S
    b_y = 2.0 * S.T @ A @ x_center + S.T @ b
    c_y = float(x_center.T @ A @ x_center + b.T @ x_center + c)

    # ------------------------------------------------------------
    # 3) Transform constraints to y-space
    # ------------------------------------------------------------

    G_y = G @ S
    r_y = r - G @ x_center

    H_y = H @ S
    h_y = h - H @ x_center

    # ------------------------------------------------------------
    # 4) Normalize constraint rows
    # ------------------------------------------------------------

    if normalize_constraints:
        G_y, r_y, G_row_norm = _normalize_constraint_rows(G_y, r_y)
        H_y, h_y, H_row_norm = _normalize_constraint_rows(H_y, h_y)
    else:
        G_row_norm = np.ones(G_y.shape[0], dtype=float)
        H_row_norm = np.ones(H_y.shape[0], dtype=float)

    # ------------------------------------------------------------
    # 5) Scale objective coefficients
    # ------------------------------------------------------------

    objective_scale = 1.0

    if scale_objective:
        objective_scale = max(
            float(np.max(np.abs(A_y))) if A_y.size > 0 else 0.0,
            float(np.max(np.abs(b_y))) if b_y.size > 0 else 0.0,
            1.0
        )

        A_y = A_y / objective_scale
        b_y = b_y / objective_scale
        c_y = c_y / objective_scale

    # ------------------------------------------------------------
    # 6) Regularize weak-cost / constraint-only variables
    # ------------------------------------------------------------

    if regularize_weak_cost:
        A_y, b_y, c_y = _add_small_regularization(
            A=A_y,
            b=b_y,
            c=c_y,
            y_ref=y0,
            eps_reg=eps_reg
        )

    # ------------------------------------------------------------
    # 7) Solve scaled QP using existing dqp()
    # ------------------------------------------------------------

    y_sol, f_y, al_iters, info = dqp(
        A=A_y,
        b=b_y,
        c=c_y,

        G=G_y,
        r=r_y,

        H=H_y,
        h=h_y,

        x0=y0,
        lb=y_lb,
        ub=y_ub,

        return_info=True,

        **dqp_kwargs
    )

    # ------------------------------------------------------------
    # 8) Map solution back to original x-space
    # ------------------------------------------------------------

    x_sol = x_center + x_scale * y_sol
    x_sol = np.clip(x_sol, lb, ub)

    f_original = float(x_sol.T @ A @ x_sol + b.T @ x_sol + c)

    if info is None:
        info = {}

    info["scaled_dqp_used"] = True
    info["scaled_solution_y"] = y_sol.copy()
    info["original_solution_x"] = x_sol.copy()

    info["scaled_objective"] = float(f_y)
    info["original_objective"] = f_original

    info["scaling_info"] = {
        "x_center": x_center.copy(),
        "x_scale": x_scale.copy(),
        "objective_scale": objective_scale,
        "G_row_norm": G_row_norm.copy(),
        "H_row_norm": H_row_norm.copy(),
        "normalize_constraints": normalize_constraints,
        "scale_objective": scale_objective,
        "regularize_weak_cost": regularize_weak_cost,
        "eps_reg": eps_reg,
    }

    info["scaled_problem"] = {
        "A_y": A_y.copy(),
        "b_y": b_y.copy(),
        "c_y": float(c_y),
        "G_y": G_y.copy(),
        "r_y": r_y.copy(),
        "H_y": H_y.copy(),
        "h_y": h_y.copy(),
        "y_lb": y_lb.copy(),
        "y_ub": y_ub.copy(),
    }

    if return_info:
        return x_sol, f_original, al_iters, info

    return x_sol, f_original, al_iters


In [7]:
# ============================================================
# LARGE-SCALE EASY QP RUNNER FOR THE FAST scaled_dqp CODE
# Inner QUBO solver: classical local-search QUBO solver
# Sizes: 50, 75, 100, 150, 200, 300
# ============================================================

import numpy as np
import pandas as pd
import time
from pathlib import Path
from scipy.optimize import minimize


# ============================================================
# USER SETTINGS
# ============================================================

N_VALUES = [50, 75, 100, 150, 200, 300]

BASE_SEED = 91000

OUTPUT_DIR = Path("scaled_dqp_easy_large_qp_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / "scaled_dqp_easy_large_qp_summary.csv"

QUALITY_TOL_PERCENT = 1.0

# Easy QP settings
LB_VALUE = -5.0
UB_VALUE = 5.0
INEQUALITY_MARGIN = 2.0

# Keep constraints not too many for large-scale validation
EQ_RATIO = 0.05
INEQ_RATIO = 0.30

# Classical local-search QUBO settings inside dqup()
CLASSICAL_NUM_RESTARTS = 80
CLASSICAL_MAX_SWEEPS = 120

# SLSQP reference settings
SLSQP_MAXITER = 3000
SLSQP_FTOL = 1e-12


# ============================================================
# BASIC HELPERS
# ============================================================

def qp_objective(A, b, c, x):
    return float(x.T @ A @ x + b.T @ x + c)


def qp_gradient(A, b, x):
    return 2.0 * A @ x + b


def feasibility_abs(G, r, H, h, x):
    if G is not None and G.shape[0] > 0:
        eq_res = G @ x - r
        eq_inf = float(np.linalg.norm(eq_res, ord=np.inf))
    else:
        eq_res = np.zeros(0)
        eq_inf = 0.0

    if H is not None and H.shape[0] > 0:
        ineq_res = H @ x - h
        ineq_viol = np.maximum(ineq_res, 0.0)
        ineq_inf = float(np.linalg.norm(ineq_viol, ord=np.inf))
    else:
        ineq_res = np.zeros(0)
        ineq_viol = np.zeros(0)
        ineq_inf = 0.0

    return eq_inf, ineq_inf, max(eq_inf, ineq_inf), eq_res, ineq_res, ineq_viol


def objective_gap_percent(f_test, f_ref):
    return 100.0 * abs(f_test - f_ref) / max(1.0, abs(f_ref))


def x_inf_gap_percent(x_test, x_ref):
    return 100.0 * np.linalg.norm(x_test - x_ref, ord=np.inf) / max(
        1.0,
        np.linalg.norm(x_ref, ord=np.inf)
    )


def x_l2_gap_percent(x_test, x_ref):
    return 100.0 * np.linalg.norm(x_test - x_ref, ord=2) / max(
        1.0,
        np.linalg.norm(x_ref, ord=2)
    )


def equality_residual_percent(G, r, x):
    if G is None or G.shape[0] == 0:
        return 0.0

    lhs = G @ x
    res = lhs - r

    denom = np.maximum.reduce([
        np.ones_like(r, dtype=float),
        np.abs(r),
        np.abs(lhs)
    ])

    return 100.0 * float(np.max(np.abs(res) / denom))


def inequality_violation_percent(H, h, x):
    if H is None or H.shape[0] == 0:
        return 0.0

    lhs = H @ x
    viol = np.maximum(lhs - h, 0.0)

    denom = np.maximum.reduce([
        np.ones_like(h, dtype=float),
        np.abs(h),
        np.abs(lhs)
    ])

    return 100.0 * float(np.max(viol / denom))


def count_total_qubo_calls(info):
    if info is None:
        return 0

    total = 0

    for dqup_info in info.get("dqup_info_history", []):
        if dqup_info is None:
            continue
        total += int(dqup_info.get("total_dvqe_calls", 0))

    return total


# ============================================================
# QP GENERATOR
# Same style as create_convex_eq_ineq_qp(...)
# ============================================================

def create_easy_large_convex_eq_ineq_qp(
    n,
    seed=1,
    lb_value=-5.0,
    ub_value=5.0,
    inequality_margin=2.0,
    eq_ratio=0.05,
    ineq_ratio=0.30
):
    """
    Generate an easy strictly feasible convex QP:

        min_x x^T A x + b^T x + c

        s.t.  Gx = r
              Hx <= h
              lb <= x <= ub

    This follows the same structure as the previous
    create_convex_eq_ineq_qp(...) generator:
    - strictly convex objective,
    - interior feasible point,
    - equality constraints generated from the feasible point,
    - inequality constraints generated with positive margin.
    """

    rng = np.random.default_rng(seed)

    # Strictly convex objective
    M = rng.normal(size=(n, n))
    A = M.T @ M + 0.5 * np.eye(n)
    A = 0.5 * (A + A.T)
    A = A / max(float(np.max(np.abs(A))), 1.0)

    b = rng.uniform(-3.0, 3.0, size=n)
    c = float(rng.uniform(-2.0, 2.0))

    # Bounds
    lb = np.full(n, lb_value, dtype=float)
    ub = np.full(n, ub_value, dtype=float)

    # Interior feasible point
    x_feasible = rng.uniform(
        low=0.25 * lb_value,
        high=0.25 * ub_value,
        size=n
    )

    # Equality constraints
    m = max(1, int(round(eq_ratio * n)))

    while True:
        G = rng.normal(size=(m, n))
        if np.linalg.matrix_rank(G) == m:
            break

    r = G @ x_feasible

    # Inequality constraints
    p = max(1, int(round(ineq_ratio * n)))

    H = rng.normal(size=(p, n))

    margin = inequality_margin + rng.uniform(
        0.0,
        inequality_margin,
        size=p
    )

    # Guarantees H @ x_feasible < h
    h = H @ x_feasible + margin

    x0_feasible = x_feasible.copy()
    x0_midpoint = 0.5 * (lb + ub)

    return (
        A,
        b,
        c,
        G,
        r,
        H,
        h,
        lb,
        ub,
        x0_feasible,
        x0_midpoint
    )


# ============================================================
# CLASSICAL CONTINUOUS QP REFERENCE
# ============================================================

def solve_slsqp_reference_qp(A, b, c, G, r, H, h, lb, ub, x0):
    """
    Solve the continuous constrained QP using scipy.optimize.minimize(SLSQP).

        min_x x^T A x + b^T x + c

        s.t.  Gx = r
              Hx <= h
              lb <= x <= ub
    """

    constraints = []

    if G is not None and G.shape[0] > 0:
        constraints.append({
            "type": "eq",
            "fun": lambda x: G @ x - r,
            "jac": lambda x: G
        })

    if H is not None and H.shape[0] > 0:
        constraints.append({
            "type": "ineq",
            "fun": lambda x: h - H @ x,
            "jac": lambda x: -H
        })

    start = time.time()

    result = minimize(
        fun=lambda x: qp_objective(A, b, c, x),
        x0=x0,
        jac=lambda x: qp_gradient(A, b, x),
        method="SLSQP",
        bounds=list(zip(lb, ub)),
        constraints=constraints,
        options={
            "ftol": SLSQP_FTOL,
            "maxiter": SLSQP_MAXITER,
            "disp": False
        }
    )

    runtime = time.time() - start

    return {
        "success": bool(result.success),
        "message": str(result.message),
        "runtime_sec": runtime,
        "x": np.asarray(result.x, dtype=float),
        "objective": float(result.fun),
        "nit": int(result.nit) if hasattr(result, "nit") else None,
    }


# ============================================================
# ONE CASE RUNNER
# ============================================================

def run_one_scaled_dqp_easy_large_qp_case(n, seed):
    print("\n" + "=" * 100)
    print(f"EASY LARGE QP WITH FAST scaled_dqp | n = {n} | seed = {seed}")
    print("=" * 100)

    (
        A,
        b,
        c,
        G,
        r,
        H,
        h,
        lb,
        ub,
        x0_feasible,
        x0_midpoint
    ) = create_easy_large_convex_eq_ineq_qp(
        n=n,
        seed=seed,
        lb_value=LB_VALUE,
        ub_value=UB_VALUE,
        inequality_margin=INEQUALITY_MARGIN,
        eq_ratio=EQ_RATIO,
        ineq_ratio=INEQ_RATIO
    )

    m = G.shape[0]
    p = H.shape[0]

    print("Problem data:")
    print("  variables:", n)
    print("  equality constraints:", m)
    print("  inequality constraints:", p)

    feas_eq_abs, feas_ineq_abs, _, _, _, _ = feasibility_abs(
        G, r, H, h, x0_feasible
    )

    mid_eq_abs, mid_ineq_abs, _, _, _, _ = feasibility_abs(
        G, r, H, h, x0_midpoint
    )

    print("\nInitial point checks:")
    print("  feasible x0 eq abs:", feas_eq_abs)
    print("  feasible x0 ineq abs:", feas_ineq_abs)
    print("  midpoint x0 eq abs:", mid_eq_abs)
    print("  midpoint x0 ineq abs:", mid_ineq_abs)

    # ------------------------------------------------------------
    # Reference solve
    # ------------------------------------------------------------

    ref = solve_slsqp_reference_qp(
        A=A,
        b=b,
        c=c,
        G=G,
        r=r,
        H=H,
        h=h,
        lb=lb,
        ub=ub,
        x0=x0_feasible
    )

    x_ref = ref["x"]
    f_ref = ref["objective"]

    ref_eq_abs, ref_ineq_abs, ref_max_abs, _, _, _ = feasibility_abs(
        G, r, H, h, x_ref
    )

    print("\nSLSQP continuous QP reference:")
    print("  success:", ref["success"])
    print("  message:", ref["message"])
    print("  time:", ref["runtime_sec"])
    print("  objective:", f_ref)
    print("  feasibility: eq =", ref_eq_abs, "| ineq =", ref_ineq_abs)

    # ------------------------------------------------------------
    # scaled_dqp solve
    # ------------------------------------------------------------

    start_dqp = time.time()

    x_dqp, f_dqp, al_iters, info = scaled_dqp(
        A=A,
        b=b,
        c=c,

        G=G,
        r=r,

        H=H,
        h=h,

        # Use feasible starting point for easy large-scale validation
        x0=x0_feasible,

        lb=lb,
        ub=ub,

        # Scaling options
        normalize_constraints=True,
        scale_objective=True,
        regularize_weak_cost=True,
        eps_reg=1e-8,

        # Augmented-Lagrangian settings
        mu0=80.0,
        mu_min=1.0,
        mu_max=1000.0,

        constraint_tol=1e-4,
        stationarity_tol=None,

        min_al_iters=1,
        max_al_iters=10,

        phr_region_tol=1e-12,
        max_region_iters=8,

        update_mu=True,
        mu_update_rule="primal_dual_balance",
        balance_factor=10.0,
        mu_increase=2.0,
        mu_decrease=1.5,
        allow_mu_decrease=False,
        primal_stall_ratio=0.90,

        # DQUP settings
        rho0=0.5,
        rho_decay=0.65,
        rho_tol=1e-8,
        obj_tol=1e-10,

        max_outer_iters=14,
        max_consecutive_outer_rejections=10,

        alpha_step_fraction=1.0,
        alpha_step_decay=0.5,
        alpha_step_tol=1e-8,

        max_inner_iters=150,
        max_consecutive_inner_rejections=15,

        alpha_warm_start=True,
        alpha_warm_start_scale=0.5,

        # Fast internal QUBO backend
        qubo_solver="classical",
        classical_num_restarts=CLASSICAL_NUM_RESTARTS,
        classical_max_sweeps=CLASSICAL_MAX_SWEEPS,

        silent=True,
        verbose=False,
        return_info=True
    )

    dqp_time = time.time() - start_dqp

    # ------------------------------------------------------------
    # Diagnostics and metrics
    # ------------------------------------------------------------

    dqp_eq_abs, dqp_ineq_abs, dqp_max_abs, _, _, _ = feasibility_abs(
        G, r, H, h, x_dqp
    )

    obj_gap_abs = abs(f_dqp - f_ref)
    x_inf_gap_abs = float(np.linalg.norm(x_dqp - x_ref, ord=np.inf))
    x_l2_gap_abs = float(np.linalg.norm(x_dqp - x_ref, ord=2))

    obj_gap_pct = objective_gap_percent(f_dqp, f_ref)
    x_inf_gap_pct = x_inf_gap_percent(x_dqp, x_ref)
    x_l2_gap_pct = x_l2_gap_percent(x_dqp, x_ref)

    eq_gap_pct = equality_residual_percent(G, r, x_dqp)
    ineq_gap_pct = inequality_violation_percent(H, h, x_dqp)
    max_constr_gap_pct = max(eq_gap_pct, ineq_gap_pct)

    quality_ok = (
        obj_gap_pct <= QUALITY_TOL_PERCENT
        and x_inf_gap_pct <= QUALITY_TOL_PERCENT
        and max_constr_gap_pct <= QUALITY_TOL_PERCENT
    )

    strict_ok = dqp_max_abs <= 1e-2

    status_history = info.get("status_history", [])
    dqp_status = status_history[-1] if len(status_history) > 0 else "completed"

    total_qubo_calls = count_total_qubo_calls(info)

    phr_region_iterations_history = info.get("phr_region_iterations_history", [])
    total_region_iterations = int(np.sum(phr_region_iterations_history)) if len(phr_region_iterations_history) > 0 else 0

    print("\nscaled_dqp result:")
    print("  status:", dqp_status)
    print("  quality ok:", quality_ok, f"({QUALITY_TOL_PERCENT:.2f}% rule)")
    print("  strict ok:", strict_ok, "(absolute residual <= 1e-2)")
    print("  time:", dqp_time)
    print("  objective:", f_dqp)
    print("  AL iterations:", al_iters)
    print("  region iterations:", total_region_iterations)
    print("  local QUBO calls:", total_qubo_calls)
    print("  feasibility: eq =", dqp_eq_abs, "| ineq =", dqp_ineq_abs)

    print("\nCompare with SLSQP reference:")
    print(f"  Objective gap:        {obj_gap_abs:.6e} ({obj_gap_pct:.6f}%)")
    print(f"  Solution inf gap:     {x_inf_gap_abs:.6e} ({x_inf_gap_pct:.6f}%)")
    print(f"  Solution L2 gap:      {x_l2_gap_abs:.6e} ({x_l2_gap_pct:.6f}%)")
    print(f"  Equality residual:    {dqp_eq_abs:.6e} ({eq_gap_pct:.6f}%)")
    print(f"  Inequality violation: {dqp_ineq_abs:.6e} ({ineq_gap_pct:.6f}%)")
    print(f"  Max constraint gap:   {dqp_max_abs:.6e} ({max_constr_gap_pct:.6f}%)")

    row = {
        "n": n,
        "seed": seed,
        "m": m,
        "p": p,

        "reference_success": bool(ref["success"]),
        "reference_message": ref["message"],
        "reference_time_sec": ref["runtime_sec"],
        "reference_objective": f_ref,
        "reference_eq_abs": ref_eq_abs,
        "reference_ineq_abs": ref_ineq_abs,

        "scaled_dqp_status": dqp_status,
        "quality_ok_1pct": bool(quality_ok),
        "strict_ok_abs_1e2": bool(strict_ok),
        "scaled_dqp_time_sec": dqp_time,
        "scaled_dqp_objective": f_dqp,
        "scaled_dqp_eq_abs": dqp_eq_abs,
        "scaled_dqp_ineq_abs": dqp_ineq_abs,
        "scaled_dqp_max_constraint_abs": dqp_max_abs,

        "objective_gap_abs": obj_gap_abs,
        "objective_gap_percent": obj_gap_pct,
        "x_inf_gap_abs": x_inf_gap_abs,
        "x_inf_gap_percent": x_inf_gap_pct,
        "x_l2_gap_abs": x_l2_gap_abs,
        "x_l2_gap_percent": x_l2_gap_pct,

        "eq_residual_percent": eq_gap_pct,
        "ineq_violation_percent": ineq_gap_pct,
        "max_constraint_gap_percent": max_constr_gap_pct,

        "al_iters": int(al_iters),
        "total_region_iterations": total_region_iterations,
        "total_qubo_calls": total_qubo_calls,

        "inner_qubo_solver": "classical_local_search",
        "classical_num_restarts": CLASSICAL_NUM_RESTARTS,
        "classical_max_sweeps": CLASSICAL_MAX_SWEEPS,
    }

    return row


# ============================================================
# MAIN RUNNER
# ============================================================

def run_scaled_dqp_easy_large_qp_tests():
    all_rows = []

    print("=" * 100)
    print("EASY LARGE-SCALE QP VALIDATION USING FAST scaled_dqp")
    print("=" * 100)
    print("Problem sizes:", N_VALUES)
    print("Reference solver: SciPy SLSQP continuous QP")
    print("Internal QUBO solver: classical local-search QUBO backend")
    print(f"Quality rule: objective, x-inf, and max constraint gaps <= {QUALITY_TOL_PERCENT}%")
    print("=" * 100)

    total_start = time.time()

    for k, n in enumerate(N_VALUES):
        seed = BASE_SEED + 1000 * k + n

        try:
            row = run_one_scaled_dqp_easy_large_qp_case(n=n, seed=seed)

        except Exception as e:
            print("\nFAILED CASE")
            print("n =", n)
            print("seed =", seed)
            print("error =", repr(e))

            row = {
                "n": n,
                "seed": seed,
                "reference_success": False,
                "scaled_dqp_status": f"failed: {repr(e)}",
                "quality_ok_1pct": False,
            }

        all_rows.append(row)

        df = pd.DataFrame(all_rows)
        df.to_csv(RESULTS_CSV, index=False)

        print("\nSaved partial results to:")
        print(RESULTS_CSV)

    total_time = time.time() - total_start

    df = pd.DataFrame(all_rows)

    print("\n" + "=" * 100)
    print("FINAL EASY LARGE-SCALE scaled_dqp SUMMARY")
    print("=" * 100)

    display_cols = [
        "n",
        "m",
        "p",
        "quality_ok_1pct",
        "objective_gap_percent",
        "x_inf_gap_percent",
        "max_constraint_gap_percent",
        "al_iters",
        "total_qubo_calls",
        "scaled_dqp_time_sec",
        "scaled_dqp_status",
    ]

    existing_cols = [col for col in display_cols if col in df.columns]
    print(df[existing_cols].to_string(index=False))

    print("\nTotal runner time:", total_time)
    print("Saved final results to:")
    print(RESULTS_CSV)

    return df




In [8]:
# ============================================================
# RUN
# ============================================================

scaled_dqp_easy_large_qp_results = run_scaled_dqp_easy_large_qp_tests()
scaled_dqp_easy_large_qp_results

EASY LARGE-SCALE QP VALIDATION USING FAST scaled_dqp
Problem sizes: [50, 75, 100, 150, 200, 300]
Reference solver: SciPy SLSQP continuous QP
Internal QUBO solver: classical local-search QUBO backend
Quality rule: objective, x-inf, and max constraint gaps <= 1.0%

EASY LARGE QP WITH FAST scaled_dqp | n = 50 | seed = 91050
Problem data:
  variables: 50
  equality constraints: 2
  inequality constraints: 15

Initial point checks:
  feasible x0 eq abs: 0.0
  feasible x0 ineq abs: 0.0
  midpoint x0 eq abs: 8.88633692658578
  midpoint x0 ineq abs: 1.1536112156761011

SLSQP continuous QP reference:
  success: True
  message: Optimization terminated successfully
  time: 0.009622812271118164
  objective: -88.4681483946248
  feasibility: eq = 1.0480505352461478e-13 | ineq = 2.1005419625907962e-13

scaled_dqp result:
  status: converged
  quality ok: True (1.00% rule)
  strict ok: True (absolute residual <= 1e-2)
  time: 189.10412549972534
  objective: -88.46841215603772
  AL iterations: 3
  regi

,n,seed,m,p,reference_success,reference_message,reference_time_sec,reference_objective,reference_eq_abs,reference_ineq_abs,...,x_l2_gap_percent,eq_residual_percent,ineq_violation_percent,max_constraint_gap_percent,al_iters,total_region_iterations,total_qubo_calls,inner_qubo_solver,classical_num_restarts,classical_max_sweeps
0,50,91050,2,15,True,Optimization terminated successfully,0.009623,-88.468148,1.048051e-13,2.100542e-13,...,0.001766,0.007708,0.014500,0.014500,3,7,1501,classical_local_search,80,120
1,75,92075,4,22,True,Optimization terminated successfully,0.042294,-162.422104,1.740830e-13,1.918465e-13,...,0.001625,0.007484,0.019727,0.019727,3,7,1245,classical_local_search,80,120
2,100,93100,5,30,True,Optimization terminated successfully,0.093521,-244.567194,9.059420e-14,4.618528e-13,...,0.001470,0.004470,0.036041,0.036041,3,6,1621,classical_local_search,80,120
3,150,94150,8,45,False,Positive directional derivative for linesearch,0.210954,-377.256376,2.702727e-12,6.609824e-12,...,0.001576,0.003838,0.013834,0.013834,3,5,1787,classical_local_search,80,120
4,200,95200,10,60,True,Optimization terminated successfully,0.556983,-502.178138,3.907985e-14,1.811884e-13,...,0.001486,0.017706,0.021689,0.021689,3,9,1121,classical_local_search,80,120
5,300,96300,15,90,False,Positive directional derivative for linesearch,2.776093,-627.808061,5.559997e-13,2.997602e-12,...,0.001698,0.036442,0.040494,0.040494,3,8,1234,classical_local_search,80,120
